# V9 MLP and LRAA adapter-off audit

This notebook performs **evaluation only**. It does not train or modify a
checkpoint. It loads the archived V9 MLP and LRAA checkpoints for seeds 13,
42, and 71, evaluates the official 3,270-example BoolQ validation split with
each adapter enabled and disabled, and writes paired on/off results.

The enabled-condition scores are first checked against the archived V9
held-out results. If that parity gate fails, the notebook stops before
reporting the new MLP-off comparison.

Input: upload `adapter-contribution-reproducibility_GitHub_upload.zip` (the
notebook also accepts the earlier V9 reproduction artifact ZIP). The base
`deberta-v3-large` model is read from Google Drive. The result ZIP downloads
automatically; the runtime is **not** deleted.

In [ ]:
# 1) Mount Google Drive before any long operation.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Install pinned dependencies quietly; print logs only if installation fails.
import subprocess, sys

packages = [
    'transformers==4.57.6',
    'huggingface-hub==0.36.2',
    'datasets==3.2.0',
    'pandas==2.2.3',
    'scikit-learn==1.6.1',
    'sentencepiece',
    'protobuf',
]
proc = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', *packages],
    text=True, capture_output=True,
)
if proc.returncode != 0:
    print(proc.stdout)
    print(proc.stderr)
    raise RuntimeError('Dependency installation failed.')
print('Dependencies ready.')

In [ ]:
# 3) Imports, GPU audit, paths, and local model copy.
import gc, hashlib, json, math, os, shutil, statistics, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU is required. In Colab select Runtime > Change runtime type > GPU.')

DEVICE = torch.device('cuda')
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('high')

DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/Colab Notebooks/NLP Modeller/deberta-v3-large')
LOCAL_MODEL_DIR = Path('/content/models/deberta-v3-large')
WORK_DIR = Path('/content/lraa_v9_mlp_adapter_off_audit')
EXTRACT_DIR = WORK_DIR / 'input_artifact'
RESULT_DIR = WORK_DIR / 'results'
for path in (WORK_DIR, EXTRACT_DIR, RESULT_DIR):
    path.mkdir(parents=True, exist_ok=True)

required_model_files = ['config.json', 'spm.model', 'tokenizer_config.json']
missing = [name for name in required_model_files if not (DRIVE_MODEL_DIR / name).is_file()]
has_weights = any((DRIVE_MODEL_DIR / name).is_file() for name in ('model.safetensors', 'pytorch_model.bin'))
if missing or not has_weights:
    raise FileNotFoundError(
        f'Incomplete model folder: {DRIVE_MODEL_DIR}; missing={missing}; weights={has_weights}'
    )
shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR, dirs_exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0))
print('Autocast dtype:', AMP_DTYPE)
print('Base model copied to local SSD.')

In [ ]:
# 4) Locate/upload the reproducibility ZIP and verify all checkpoint hashes.
from google.colab import files

candidates = list(Path('/content').glob('*adapter*contribution*zip'))
candidates += list(Path('/content').glob('*v9*reproduction*zip'))
candidates += list(Path('/content/drive/MyDrive').rglob('*adapter*contribution*zip'))
candidates += list(Path('/content/drive/MyDrive').rglob('*v9*reproduction*zip'))
if candidates:
    ARTIFACT_ZIP = sorted(candidates)[0]
    print('Using:', ARTIFACT_ZIP)
else:
    print('Select adapter-contribution-reproducibility_GitHub_upload.zip')
    uploaded = files.upload()
    if not uploaded:
        raise FileNotFoundError('No artifact ZIP uploaded.')
    ARTIFACT_ZIP = Path('/content') / next(iter(uploaded))

with zipfile.ZipFile(ARTIFACT_ZIP) as zf:
    bad = zf.testzip()
    if bad:
        raise RuntimeError(f'Corrupt member in ZIP: {bad}')
    zf.extractall(EXTRACT_DIR)

manifest_candidates = []
for path in EXTRACT_DIR.rglob('checkpoint_manifest.csv'):
    checkpoint_dir = path.parent if path.parent.name == 'checkpoints' else path.parent / 'checkpoints'
    if checkpoint_dir.is_dir() and len(list(checkpoint_dir.glob('*.pt'))) >= 9:
        manifest_candidates.append((path, checkpoint_dir))
if not manifest_candidates:
    raise FileNotFoundError('Could not locate a checkpoint manifest with nine checkpoint files.')
MANIFEST_PATH, CHECKPOINT_DIR = sorted(manifest_candidates, key=lambda x: len(x[0].parts))[0]

manifest = pd.read_csv(MANIFEST_PATH)
if len(manifest) != 9:
    raise ValueError(f'Expected nine checkpoints, found {len(manifest)}')
for row in manifest.itertuples(index=False):
    path = CHECKPOINT_DIR / row.filename
    actual = hashlib.sha256(path.read_bytes()).hexdigest()
    if path.stat().st_size != int(row.bytes) or actual != row.sha256:
        raise RuntimeError(f'Checkpoint manifest mismatch: {row.filename}')

run_result_candidates = list(EXTRACT_DIR.rglob('table_run_results.csv'))
if not run_result_candidates:
    raise FileNotFoundError('Archived table_run_results.csv not found.')
RUN_RESULTS_PATH = sorted(run_result_candidates, key=lambda p: len(p.parts))[0]
archived_runs = pd.read_csv(RUN_RESULTS_PATH)
required_models = {'mlp_adapter', 'lcca_adapter'}
if not required_models.issubset(set(archived_runs['model'])):
    raise ValueError(f'Archived run table lacks required models: {required_models}')

print('Checkpoint hashes verified: 9/9')
print('Checkpoint directory:', CHECKPOINT_DIR)
print('Archived held-out table:', RUN_RESULTS_PATH)

In [ ]:
# 5) Load and tokenize the official BoolQ validation split using exact V9 serialization.
SEEDS = (13, 42, 71)
MAX_LENGTH = 512
EVAL_BATCH_SIZE = 4
LATENT_RANK = 256
LATENT_HEADS = 8
MLP_BOTTLENECK = 512

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

heldout = load_dataset('google/boolq', split='validation', token=HF_TOKEN)
if len(heldout) != 3270:
    raise ValueError(f'Expected 3,270 held-out examples, found {len(heldout)}')

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR, token=HF_TOKEN, use_fast=True)

def encode_batch(batch):
    prefixes = [f'question: {q} answer:' for q in batch['question']]
    yes = tokenizer(
        [q + ' yes' for q in prefixes], batch['passage'],
        truncation='only_second', max_length=MAX_LENGTH,
    )
    no = tokenizer(
        [q + ' no' for q in prefixes], batch['passage'],
        truncation='only_second', max_length=MAX_LENGTH,
    )
    return {
        'yes_input_ids': yes['input_ids'],
        'yes_attention_mask': yes['attention_mask'],
        'no_input_ids': no['input_ids'],
        'no_attention_mask': no['attention_mask'],
    }

encoded = heldout.map(
    encode_batch, batched=True, batch_size=256,
    remove_columns=['question', 'passage'], desc='Exact V9 held-out tokenization',
)

class PairDataset(Dataset):
    def __init__(self, dataset): self.dataset = dataset
    def __len__(self): return len(self.dataset)
    def __getitem__(self, idx):
        row = self.dataset[int(idx)]
        return {
            'candidates': [
                {'input_ids': row['yes_input_ids'], 'attention_mask': row['yes_attention_mask']},
                {'input_ids': row['no_input_ids'], 'attention_mask': row['no_attention_mask']},
            ],
            'label': 0 if bool(row['answer']) else 1,
            'gold': int(bool(row['answer'])),
        }

def collate(batch):
    flat = [candidate for item in batch for candidate in item['candidates']]
    padded = tokenizer.pad(flat, padding=True, return_tensors='pt')
    bsz = len(batch)
    return {
        'input_ids': padded['input_ids'].view(bsz, 2, -1),
        'attention_mask': padded['attention_mask'].view(bsz, 2, -1),
        'labels': torch.tensor([item['label'] for item in batch], dtype=torch.long),
        'gold': torch.tensor([item['gold'] for item in batch], dtype=torch.long),
    }

loader = DataLoader(
    PairDataset(encoded), batch_size=EVAL_BATCH_SIZE, shuffle=False,
    collate_fn=collate, num_workers=2, pin_memory=True,
)
print('Held-out examples ready:', len(encoded))

In [ ]:
# 6) Exact V9 MLP/LRAA definitions and checkpoint loading.
class GatedMLPAdapter(nn.Module):
    def __init__(self, hidden_size, bottleneck, dropout=0.1):
        super().__init__()
        self.down = nn.Linear(hidden_size, bottleneck, bias=False)
        self.up = nn.Linear(bottleneck, hidden_size, bias=False)
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.alpha_logit = nn.Parameter(torch.tensor(math.log(0.1 / 0.9), dtype=torch.float32))
    @property
    def alpha(self): return torch.sigmoid(self.alpha_logit)
    def forward(self, hidden, attention_mask=None, enabled=True):
        if not enabled: return hidden
        z = self.up(F.gelu(self.down(hidden)))
        return self.norm(hidden + self.alpha * self.dropout(z))

class LatentRelationAdapter(nn.Module):
    def __init__(self, hidden_size, rank, heads, dropout=0.1):
        super().__init__()
        assert rank % heads == 0
        self.rank, self.heads, self.head_dim = rank, heads, rank // heads
        self.q = nn.Linear(hidden_size, rank, bias=False)
        self.k = nn.Linear(hidden_size, rank, bias=False)
        self.v = nn.Linear(hidden_size, rank, bias=False)
        self.out = nn.Linear(rank, hidden_size, bias=False)
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.alpha_logit = nn.Parameter(torch.tensor(math.log(0.1 / 0.9), dtype=torch.float32))
    @property
    def alpha(self): return torch.sigmoid(self.alpha_logit)
    def forward(self, hidden, attention_mask, enabled=True):
        if not enabled: return hidden
        bsz, seq_len, _ = hidden.shape
        def split(x): return x.view(bsz, seq_len, self.heads, self.head_dim).transpose(1, 2)
        q, k, v = split(self.q(hidden)), split(self.k(hidden)), split(self.v(hidden))
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(
            attention_mask[:, None, None, :] == 0,
            torch.finfo(scores.dtype).min,
        )
        attn = torch.softmax(scores, dim=-1)
        z = torch.matmul(attn, v).transpose(1, 2).contiguous().view(bsz, seq_len, self.rank)
        return self.norm(hidden + self.alpha * self.dropout(self.out(z)))

class FrozenFeatureScorer(nn.Module):
    def __init__(self, mode, hidden_size):
        super().__init__()
        if mode == 'mlp': self.adapter = GatedMLPAdapter(hidden_size, MLP_BOTTLENECK)
        elif mode == 'lcca': self.adapter = LatentRelationAdapter(hidden_size, LATENT_RANK, LATENT_HEADS)
        else: raise ValueError(mode)
        self.dropout = nn.Dropout(0.15)
        self.scorer = nn.Linear(hidden_size, 1)
    def forward(self, hidden, attention_mask, adapter_enabled=True):
        hidden = self.adapter(hidden, attention_mask, enabled=adapter_enabled)
        return self.scorer(self.dropout(hidden[:, 0]))

def safe_load(path):
    try: return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError: return torch.load(path, map_location='cpu')

encoder = AutoModel.from_pretrained(LOCAL_MODEL_DIR, token=HF_TOKEN).to(DEVICE).eval()
for parameter in encoder.parameters(): parameter.requires_grad = False
hidden_size = int(encoder.config.hidden_size)

models = {}
for alias in ('mlp', 'lcca'):
    for seed in SEEDS:
        checkpoint = safe_load(CHECKPOINT_DIR / f'{alias}_seed{seed}.pt')
        model = FrozenFeatureScorer(alias, hidden_size).to(DEVICE)
        model.adapter.load_state_dict(checkpoint['adapter_state_dict'], strict=True)
        model.scorer.load_state_dict(checkpoint['scorer_state_dict'], strict=True)
        model.eval()
        models[(alias, seed)] = model
print('Loaded trained models:', len(models))

In [ ]:
# 7) One shared encoder pass per batch; evaluate every model with adapter on and off.
variants = {
    (alias, seed, state): (model, state == 'on')
    for (alias, seed), model in models.items()
    for state in ('on', 'off')
}
totals = {key: {'gold': [], 'pred': [], 'loss_sum': 0.0} for key in variants}
start_time = time.time()

with torch.no_grad():
    for batch_i, batch in enumerate(loader, 1):
        ids = batch['input_ids'].to(DEVICE, non_blocking=True)
        mask = batch['attention_mask'].to(DEVICE, non_blocking=True)
        labels = batch['labels'].to(DEVICE, non_blocking=True)
        gold = batch['gold'].numpy()
        bsz, nopt, seq_len = ids.shape
        flat_ids = ids.view(bsz * nopt, seq_len)
        flat_mask = mask.view(bsz * nopt, seq_len)
        with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            hidden = encoder(input_ids=flat_ids, attention_mask=flat_mask).last_hidden_state
            for key, (model, enabled) in variants.items():
                logits = model(hidden, flat_mask, adapter_enabled=enabled).view(bsz, nopt)
                loss = F.cross_entropy(logits.float(), labels)
                # candidate index 0=yes, 1=no
                pred = (logits.argmax(dim=-1) == 0).long().cpu().numpy()
                totals[key]['gold'].extend(gold.tolist())
                totals[key]['pred'].extend(pred.tolist())
                totals[key]['loss_sum'] += loss.item() * bsz
        if batch_i == 1 or batch_i % 100 == 0 or batch_i == len(loader):
            print(f'Batches: {batch_i}/{len(loader)}')

def metric_row(alias, seed, state, values):
    gold = np.asarray(values['gold'], dtype=int)
    pred = np.asarray(values['pred'], dtype=int)
    return {
        'model': alias,
        'seed': seed,
        'adapter_state': state,
        'loss': values['loss_sum'] / len(gold),
        'accuracy': accuracy_score(gold, pred),
        'balanced_accuracy': balanced_accuracy_score(gold, pred),
        'macro_f1': f1_score(gold, pred, average='macro', zero_division=0),
        'predicted_yes_rate': pred.mean(),
        'true_yes_rate': gold.mean(),
    }, gold, pred

run_rows, prediction_rows = [], []
for (alias, seed, state), values in totals.items():
    row, gold, pred = metric_row(alias, seed, state, values)
    run_rows.append(row)
    for idx, (g, p) in enumerate(zip(gold, pred)):
        prediction_rows.append({
            'model': alias, 'seed': seed, 'adapter_state': state,
            'example_index': idx, 'gold': int(g), 'prediction': int(p),
        })

run_df = pd.DataFrame(run_rows).sort_values(['model', 'seed', 'adapter_state']).reset_index(drop=True)
prediction_df = pd.DataFrame(prediction_rows)
run_df.to_csv(RESULT_DIR / 'adapter_on_off_run_results.csv', index=False)
prediction_df.to_csv(RESULT_DIR / 'adapter_on_off_predictions.csv', index=False)
print('Evaluation minutes:', round((time.time() - start_time) / 60, 2))
display(run_df)

In [ ]:
# 8) Enabled-condition parity gate against archived V9 held-out results.
archived_alias = {'mlp_adapter': 'mlp', 'lcca_adapter': 'lcca'}
parity_rows = []
for archived_model, alias in archived_alias.items():
    expected_model = archived_runs[archived_runs.model == archived_model].set_index('seed')
    actual_model = run_df[(run_df.model == alias) & (run_df.adapter_state == 'on')].set_index('seed')
    for seed in SEEDS:
        for archived_col, actual_col in [
            ('normal_accuracy', 'accuracy'),
            ('normal_balanced_accuracy', 'balanced_accuracy'),
            ('normal_macro_f1', 'macro_f1'),
        ]:
            expected = float(expected_model.loc[seed, archived_col])
            actual = float(actual_model.loc[seed, actual_col])
            difference_points = 100 * (actual - expected)
            parity_rows.append({
                'model': alias, 'seed': seed, 'metric': actual_col,
                'archived_percent': 100 * expected,
                'recalculated_percent': 100 * actual,
                'difference_points': difference_points,
                'pass': abs(difference_points) <= 0.05,
            })

parity_df = pd.DataFrame(parity_rows)
parity_df.to_csv(RESULT_DIR / 'enabled_condition_parity_audit.csv', index=False)
if not parity_df['pass'].all():
    display(parity_df[~parity_df['pass']])
    raise RuntimeError('ENABLED-CONDITION PARITY FAILED. Adapter-off interpretation intentionally stopped.')
print('ENABLED-CONDITION PARITY PASS:', int(parity_df['pass'].sum()), '/', len(parity_df))

In [ ]:
# 9) Paired on-minus-off summaries, seed intervals, and held-out McNemar tests.
T_CRIT_DF2 = 4.302652729911275

def exact_mcnemar_p(a_only, b_only):
    n = a_only + b_only
    if n == 0: return 1.0
    k = min(a_only, b_only)
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / (2 ** n))

summary_rows, mcnemar_rows = [], []
gain_vectors = {}
for alias in ('mlp', 'lcca'):
    model_runs = run_df[run_df.model == alias].set_index(['seed', 'adapter_state'])
    for metric in ('accuracy', 'balanced_accuracy', 'macro_f1'):
        gains = np.array([
            100 * (model_runs.loc[(seed, 'on'), metric] - model_runs.loc[(seed, 'off'), metric])
            for seed in SEEDS
        ])
        gain_vectors[(alias, metric)] = gains
        mean = float(gains.mean())
        sd = float(gains.std(ddof=1))
        half = T_CRIT_DF2 * sd / math.sqrt(len(gains))
        summary_rows.append({
            'model': alias, 'metric': metric,
            'on_mean_percent': 100 * model_runs.xs('on', level='adapter_state')[metric].mean(),
            'off_mean_percent': 100 * model_runs.xs('off', level='adapter_state')[metric].mean(),
            'on_minus_off_points': mean,
            'gain_sd_points': sd,
            'ci95_low_points': mean - half,
            'ci95_high_points': mean + half,
            'positive_seeds': int((gains > 0).sum()),
        })

    for seed in SEEDS:
        on = prediction_df[(prediction_df.model == alias) & (prediction_df.seed == seed) & (prediction_df.adapter_state == 'on')]
        off = prediction_df[(prediction_df.model == alias) & (prediction_df.seed == seed) & (prediction_df.adapter_state == 'off')]
        merged = on.merge(off, on=['model', 'seed', 'example_index', 'gold'], suffixes=('_on', '_off'))
        on_correct = merged.prediction_on.to_numpy() == merged.gold.to_numpy()
        off_correct = merged.prediction_off.to_numpy() == merged.gold.to_numpy()
        on_only = int(np.sum(on_correct & ~off_correct))
        off_only = int(np.sum(~on_correct & off_correct))
        mcnemar_rows.append({
            'model': alias, 'seed': seed,
            'on_only_correct': on_only,
            'off_only_correct': off_only,
            'exact_two_sided_p': exact_mcnemar_p(on_only, off_only),
        })

summary_df = pd.DataFrame(summary_rows)
mcnemar_df = pd.DataFrame(mcnemar_rows)

# The reviewer-facing estimand: is the on-minus-off collapse larger for LRAA
# than for the parameter-matched MLP? This prevents either ablation from being
# presented in isolation as topology-specific evidence.
difference_rows = []
for metric in ('accuracy', 'balanced_accuracy', 'macro_f1'):
    differences = gain_vectors[('lcca', metric)] - gain_vectors[('mlp', metric)]
    mean = float(differences.mean())
    sd = float(differences.std(ddof=1))
    half = T_CRIT_DF2 * sd / math.sqrt(len(differences))
    difference_rows.append({
        'metric': metric,
        'lraa_gain_minus_mlp_gain_points': mean,
        'seed_sd_points': sd,
        'ci95_low_points': mean - half,
        'ci95_high_points': mean + half,
        'positive_seeds': int((differences > 0).sum()),
        **{f'seed_{seed}_points': float(value) for seed, value in zip(SEEDS, differences)},
    })
difference_df = pd.DataFrame(difference_rows)

summary_df.to_csv(RESULT_DIR / 'adapter_on_off_summary.csv', index=False)
mcnemar_df.to_csv(RESULT_DIR / 'adapter_on_off_mcnemar.csv', index=False)
difference_df.to_csv(RESULT_DIR / 'difference_of_adapter_gains.csv', index=False)

display(summary_df)
display(mcnemar_df)
display(difference_df)

lines = [
    '| Model | Metric | On (%) | Off (%) | On−off (points) | 95% seed CI | Positive seeds |',
    '|---|---|---:|---:|---:|---:|:---:|',
]
for row in summary_rows:
    lines.append(
        f"| {row['model'].upper()} | {row['metric'].replace('_', ' ')} | "
        f"{row['on_mean_percent']:.2f} | {row['off_mean_percent']:.2f} | "
        f"{row['on_minus_off_points']:+.2f} | "
        f"[{row['ci95_low_points']:+.2f}, {row['ci95_high_points']:+.2f}] | "
        f"{row['positive_seeds']}/3 |"
    )
markdown_table = '\n'.join(lines)
(RESULT_DIR / 'adapter_on_off_table.md').write_text(markdown_table + '\n', encoding='utf-8')
print(markdown_table)

In [ ]:
# 10) Save exact environment/configuration, package results, and download automatically.
environment = {
    'analysis': 'V9 held-out MLP/LRAA adapter-on versus adapter-off audit',
    'training_performed': False,
    'heldout_examples': len(encoded),
    'seeds': list(SEEDS),
    'base_model_directory': str(DRIVE_MODEL_DIR),
    'input_artifact_zip': str(ARTIFACT_ZIP),
    'input_checkpoint_manifest_sha256': hashlib.sha256(MANIFEST_PATH.read_bytes()).hexdigest(),
    'gpu': torch.cuda.get_device_name(0),
    'amp_dtype': str(AMP_DTYPE),
    'torch_version': torch.__version__,
    'transformers_version': __import__('transformers').__version__,
    'datasets_version': __import__('datasets').__version__,
    'enabled_condition_parity_passed': True,
    'parity_tolerance_points': 0.05,
}
(RESULT_DIR / 'run_environment.json').write_text(json.dumps(environment, indent=2), encoding='utf-8')
shutil.copy2(MANIFEST_PATH, RESULT_DIR / 'checkpoint_manifest.csv')

file_rows = []
for path in sorted(RESULT_DIR.iterdir()):
    if path.is_file() and path.name != 'result_file_manifest_sha256.csv':
        file_rows.append({
            'filename': path.name,
            'bytes': path.stat().st_size,
            'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
        })
pd.DataFrame(file_rows).to_csv(RESULT_DIR / 'result_file_manifest_sha256.csv', index=False)

zip_base = '/content/lraa_v9_mlp_adapter_off_audit_results'
zip_path = shutil.make_archive(zip_base, 'zip', RESULT_DIR)
with zipfile.ZipFile(zip_path) as zf:
    bad = zf.testzip()
    if bad: raise RuntimeError(f'Output ZIP failed integrity check: {bad}')
print('Created:', zip_path)
files.download(zip_path)